In [ ]:
import requests as req
from bs4 import BeautifulSoup
import pandas as pd
import re


source = {}

master_data = {}

def parse_sources():

    with open("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/raw/links_for_military_data.txt", "r") as file:
        lines = file.readlines()


    sources = False
    for line in lines:
        line = line.strip()
        if line.startswith("other_sources"):
            sources = True
            continue
        elif line == "}":
            break
        elif sources and line:

            line = line.rstrip(",")

            if ": " in line:
                key, value = line.split(": ", 1)

                key = key.strip().strip("'").strip('"')
                value = value.strip().strip("'").strip('"')
                source[key] = value

parse_sources()

# print(source)


In [ ]:
def get_links():

    for key,val in source.items():
        response = req.get(key)

        # file_name = "/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/raw/" + val + ".html"

        # file = open(file_name,"w+")
        # file.write(response.text)
        # file.close()

        soup = BeautifulSoup(response.text,"html.parser")

        rec = soup.find_all("div",class_="recordsetContainer")

        for r in rec:

             cnt_shrtname = r.select_one("div.shortFormName span").get_text(strip=True)
             cnt_longname = r.select_one("div.longFormName span").get_text(strip=True)
             cnt_val = r.select_one("div.valueContainer span span").get_text(strip=True)
             cnt_val = re.sub(r"[^0-9]","",cnt_val)
             cnt_val = "".join(cnt_val.split())
             cnt_val = int(cnt_val) if cnt_val.isdigit() else cnt_val
             if cnt_longname not in master_data:
                 master_data[cnt_longname] = {
                     "shortname": cnt_shrtname
                 }

             master_data[cnt_longname][val] = cnt_val

get_links()

# print(len(master_data))

# for key,val in master_data.items():
#     print(key,val)

working of `get_links` function:

*   **Iterates through Sources**: It loops through each url (key) and its associated data category (value) in the `source` dictionary.
*   **Fetches Webpage Content**: For every URL, it sends an HTTP GET request to retrieve the webpage's HTML content.
*   **Parses HTML**: It uses BeautifulSoup to parse the fetched HTML, making it easy to extract specific data elements.
*   **Identifies Data Records**: It searches for all `div` elements that have the class `recordsetContainer`. Each of these `div`s is expected to contain data for a specific country.
*   **Extracts Country-Specific Data**: For each identified record container, it extracts three pieces of information:
    *   `cnt_shrtname`: The country's abbreviated name (e.g., "USA").
    *   `cnt_longname`: The full name of the country (e.g., "United States").
    *   `cnt_val`: A numerical value corresponding to the data category for that country. It cleans this value by removing any non-digit characters (like newlines, tabs, "km", commas) and then converts it to an integer. If the cleaned value isn't purely numeric, it defaults to `0`.
*   **Populates `master_data`**: It then organizes this extracted information into the `master_data` dictionary:
    *   If a country's `cnt_longname` is not yet a key in `master_data`, a new entry is created for that country, including its `shortname`.
    *   Finally, the `cnt_val` is added to the respective country's dictionary within `master_data`, using the data category (from the `source` dictionary's `val`) as the key.

In [ ]:
dataframe = pd.DataFrame.from_dict(master_data,orient="index")
dataframe.fillna(0,inplace=True)

dataframe.reset_index(inplace=True)
dataframe.rename(columns={"index":"country"},inplace=True)
# print(dataframe.shape)

# print(dataframe.describe())
# print(dataframe.isna())
dataframe.head()

,country,shortname,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,China,CHN,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,...,225341000000,366160000000,6654000000000,4827000000,5313000000,143197000000,9596960,14500,22457,27700
1,India,IND,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,...,33170000000,58867000000,1381000000000,985671000,1200000000,111052000000,3287263,7000,13888,14500
2,United States,USA,341963408,150463900,124816644,4445524,1328000,799500,0,13043,...,1029000000000,914301000000,13402000000000,548849000,476044000,248941000000,9833517,19924,12002,41009
3,Indonesia,INO,281562465,137965608,114595923,4786562,400000,400000,250000,459,...,57410000000,36061000000,1408000000000,659357000,202283000,34869000000,1904569,54716,2958,21579
4,Pakistan,PAK,252363571,108516336,85803614,4794908,654000,550000,500000,1399,...,36937000000,46448000000,592219000000,12712000,34027000,3064000000,796095,1046,7257,0


In [ ]:
print(dataframe.describe())

In [ ]:
print(dataframe.isna())

In [ ]:
dataframe.to_csv("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/processeddata/unified_military_data.csv",index=False)

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/unified military analytics and visualization(infosys internship project)/project/data/processeddata/unified_military_data.csv")
df.head()

,country,shortname,total_population,total_military_manpower,fit_for_service,population_reaching_military_age_annually,active_personnel,reserve_personnel,paramilitary,total_military_aircraft,...,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,coastline_coverage_km,border_coverage_km,waterway_coverage_km
0,China,CHN,1415043270,764123366,626864169,19810606,2035000,510000,625000,3309,...,225341000000,366160000000,6654000000000,4827000000,5313000000,143197000000,9596960,14500,22457,27700
1,India,IND,1409128296,662290299,522786598,23955181,1455550,1155000,2527000,2229,...,33170000000,58867000000,1381000000000,985671000,1200000000,111052000000,3287263,7000,13888,14500
2,United States,USA,341963408,150463900,124816644,4445524,1328000,799500,0,13043,...,1029000000000,914301000000,13402000000000,548849000,476044000,248941000000,9833517,19924,12002,41009
3,Indonesia,INO,281562465,137965608,114595923,4786562,400000,400000,250000,459,...,57410000000,36061000000,1408000000000,659357000,202283000,34869000000,1904569,54716,2958,21579
4,Pakistan,PAK,252363571,108516336,85803614,4794908,654000,550000,500000,1399,...,36937000000,46448000000,592219000000,12712000,34027000,3064000000,796095,1046,7257,0
